In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [19]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
import os
from dotenv import load_dotenv
import copy
load_dotenv()

# Load a sample dataset
from datasets import load_dataset

True

In [3]:
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [5]:
# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

In [6]:
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

In [7]:
tokenizer

GPT2TokenizerFast(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=

Generate with the base model
Here we will try out the base model which does not have a chat template.

In [8]:
# Let's test the base model before training
prompt = "Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print(f"Formatted prompt: {formatted_prompt}")

Formatted prompt: <|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>



In [9]:
# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=30)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Before training:
<|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>

The salt and sugar are both salts, but they are not the same. Salt is a mineral, while sugar is a carbohydrate. Salt is a


In [10]:
# Use a reasoning dataset
ds = load_dataset("prithivMLmods/Deepthink-Reasoning")

ds

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 251
    })
})

In [11]:
ds["train"].shuffle().select([1])[:1]

{'prompt': ['How do Insertion sort, Heapsort, Quicksort, and Merge sort work?'],
 'response': ['<|thinking|>\n## Analyzing the request\nThe user wants explanations of how Insertion Sort, Heapsort, Quicksort, and Merge Sort work.\n\n## Planning the solution\n1. Provide a brief overview of each sorting algorithm.\n2. Explain the step-by-step process of each algorithm.\n3. Highlight the key characteristics and use cases of each algorithm.\n\n## Deciding on the approach\nI will write a structured response that:\n1. Introduces each sorting algorithm.\n2. Explains the working mechanism of each algorithm.\n3. Summarizes the key points for clarity.\n\n## Considering edge cases\n- Ensure the explanations are clear and concise.\n- Highlight the differences and use cases of each algorithm.\n</|thinking|>\n<|actual_response|>\n### Insertion Sort\n**Overview**: Insertion Sort is a simple sorting algorithm that builds the final sorted array one element at a time.\n\n**How It Works**:\n1. Start with 

In [12]:
def tokenize_function(examples):
    prompts = [p.strip() for p in examples["prompt"]]
    responses = [r.strip() for r in examples["response"]]
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}, {"role": "assistant", "content": r}],
            tokenize=False
        )
        for p, r in zip(prompts, responses)
    ]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=512, )

ds = ds.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing dataset",
)

Tokenizing dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

In [14]:
ds["train"][2]

{'prompt': 'Create a Python script to print the numbers from 1 to 50, but for multiples of 3 print "Fizz" instead of the number, for multiples of 5 print "Buzz" instead of the number and for multiples of both 3 and 5 print "FizzBuzz".\n\nNot applicable',
 'response': '<|thinking|>\n## Thinking about the FizzBuzz problem\n\nThe problem is a classic coding challenge.\n\n## Key Steps\n\n1. **Loop through numbers 1 to 50:** A `for` loop will be used to iterate through the numbers.\n2. **Check for multiples of 3 and 5:** The modulo operator (`%`) will be used to check if a number is divisible by 3 or 5.\n3. **Conditional Printing:** `if`, `elif`, and `else` statements will be used to print "Fizz", "Buzz", "FizzBuzz", or the number itself based on the conditions.\n\n## Python Code\n\n```python\ndef fizzbuzz():\n  """Prints numbers from 1 to 50, replacing multiples of 3 with "Fizz",\n  multiples of 5 with "Buzz", and multiples of both with "FizzBuzz".\n  """\n  for i in range(1, 51):\n    if 

In [15]:
finetune_name = "SmolLM2-FT-MyDataset"

# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=400,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=20,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    # eval_strategy="steps",  # Evaluate the model at regular intervals
    # eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
   
)

# create a copy of the model to avoid modifying the original
copy_model = copy.deepcopy(model)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=copy_model,
    args=sft_config,
    train_dataset=ds["train"],
   #  eval_dataset=ds["train"],
)


/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/transformers/training_args.py:2214: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(


Truncating train dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

In [16]:
trainer.train()

/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,1.440800
40,1.255000
60,1.014700
80,0.920700
100,0.864400
120,0.801200
140,0.721800
160,0.621400
180,0.720900
200,0.657500


/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=400, training_loss=0.6962287378311157, metrics={'train_runtime': 316.6536, 'train_samples_per_second': 5.053, 'train_steps_per_second': 1.263, 'total_flos': 520053684830208.0, 'train_loss': 0.6962287378311157})

In [ ]:
# Let's test the base model before training
prompt = "Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?"

formatted_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], tokenize=False
)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# Create a streamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)


outputs = model.generate(**inputs, max_new_tokens=200, stream_output=True)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("\n\n ====== Training complete, generating new outputs ======")

outputs = trainer.model.generate(**inputs, max_new_tokens=200, stream_output=True)
print("After training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

ValueError: The following `model_kwargs` are not used by the model: ['stream_output'] (note: typos in the generate arguments will also show up in this list)